In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
import plotly.express as px
from sklearn.metrics import mean_squared_error
from sklearn.utils import shuffle
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn import linear_model


In [2]:
import seaborn as sns

In [3]:
import os

In [4]:
autodf = pd.read_csv('data/vehicles.csv')

In [5]:
autodf.columns

Index(['id', 'region', 'price', 'year', 'manufacturer', 'model', 'condition',
       'cylinders', 'fuel', 'odometer', 'title_status', 'transmission', 'VIN',
       'drive', 'size', 'type', 'paint_color', 'state'],
      dtype='str')

In [6]:
autodf.dropna(inplace=True) # cleaning, dropping NaN

In [7]:
autodf = autodf[autodf['price'] >= 500] # remove cars with price of zero

Clearly a U.S. auto sales.  The top four are ford, chevrolet, toyota  and Honda. There are 11 Toyota manufacturing plants, and 12 Honda manufacturing plants in the U.S.

The id, and VIN colums are not helpful for creating a model.

In [8]:

# Drop columns that won't help regression
df = autodf.drop(columns=['id', 'VIN'])  # IDs are meaningless for regression


The rest of the values are string values. They can be divided into two categories.  Those that depict a value where order matters, and those that are categories where we can use one shot encoding.

In [9]:
conditions =  {'salvage': 0, 'fair': 1, 'good': 2, 'excellent': 3, 'like new': 4, 'new': 5}
cylinders ={'3 cylinders': 3, '4 cylinders': 4, '5 cylinders': 5,
                  '6 cylinders': 6, '8 cylinders': 8, '10 cylinders': 10, '12 cylinders': 12}
sizes =  {'sub-compact': 0, 'compact': 1, 'mid-size': 2, 'full-size': 3}


In [10]:
ordinal_map = {
    'condition': conditions,
    'cylinders': cylinders,
    'size':      sizes,
}

In [11]:
for col, mapping in ordinal_map.items():
    df[col] = df[col].map(mapping)

In [12]:
df[['size', 'cylinders', 'condition']].head()

,size,cylinders,condition
215,1,4.0,3
219,2,6.0,1
268,1,4.0,3
337,3,6.0,3
338,3,6.0,1


These are symbolic values where the order does not matter. Use 'get_dummies, for "one shot encoding"

In [13]:
# --- One-hot encoding (no natural order) ---
nominal_cols = ['region', 'manufacturer', 'model', 'fuel',
                'title_status', 'transmission', 'drive',
                'type', 'paint_color', 'state']

df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

In [14]:
df.head()

,price,year,condition,cylinders,odometer,size,region_abilene,region_akron / canton,region_albany,region_albuquerque,...,state_sd,state_tn,state_tx,state_ut,state_va,state_vt,state_wa,state_wi,state_wv,state_wy
215,4000,2002.0,3,4.0,155000.0,1,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
219,2500,1995.0,1,6.0,110661.0,2,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
268,9000,2008.0,3,4.0,56700.0,1,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
337,8950,2011.0,3,6.0,164000.0,3,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
338,4000,1972.0,1,6.0,88100.0,3,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [15]:
nan_count = df.isna().sum().sum()
if nan_count > 0:
    df = df.dropna()
nan_count = df.isna().sum().sum()
nan_count

np.int64(0)

### We have no Nans, so we are good to go!

In [16]:
# --- Ready for regression ---
y = df['price']
X = df.drop(columns=['price'])


In [17]:
X.head(3)

,year,condition,cylinders,odometer,size,region_abilene,region_akron / canton,region_albany,region_albuquerque,region_altoona-johnstown,...,state_sd,state_tn,state_tx,state_ut,state_va,state_vt,state_wa,state_wi,state_wv,state_wy
215,2002.0,3,4.0,155000.0,1,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
219,1995.0,1,6.0,110661.0,2,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
268,2008.0,3,4.0,56700.0,1,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


This is a bad mse, but we see the selected values = year, odometer, fuel_gas, and four wheel drive.
Are four wheel drive all trucks?  This is consitent with the observation that the top 9 auto are all trucks

In [18]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Lasso, LassoCV

In [19]:
X = df.drop('price', axis = 1)
y = df.price

In [20]:
# --- Scale features 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [21]:

# --- LassoCV auto-finds the best alpha ---
lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_scaled, y)


,"eps eps: float, default=1e-3Length of the path. ``eps=1e-3`` means that``alpha_min / alpha_max = 1e-3``.",0.001
,"n_alphas n_alphas: int, default=100Number of alphas along the regularization path... deprecated:: 1.7 `n_alphas` was deprecated in 1.7 and will be removed in 1.9. Use `alphas` instead.",'deprecated'
,"alphas alphas: array-like or int, default=NoneValues of alphas to test along the regularization path.If int, `alphas` values are generated automatically.If array-like, list of alpha values to use... versionchanged:: 1.7 `alphas` accepts an integer value which removes the need to pass `n_alphas`... deprecated:: 1.7 `alphas=None` was deprecated in 1.7 and will be removed in 1.9, at which point the default value will be set to 100.",'warn'
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: 'auto', bool or array-like of shape (n_features, n_features), default='auto'Whether to use a precomputed Gram matrix to speed upcalculations. If set to ``'auto'`` let us decide. The Grammatrix can also be passed as argument.",'auto'
,"max_iter max_iter: int, default=1000The maximum number of iterations.",10000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``.",0.0001
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"cv cv: int, cross-validation generator or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- int, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For int/None inputs, :class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: bool or int, default=FalseAmount of verbosity.",False
,"n_jobs n_jobs: int, default=NoneNumber of CPUs to use during the cross validation.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [22]:
# --- Extract coefficients ---
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': lasso.coef_
}).sort_values('coefficient', key=abs, ascending=False)


In [23]:
coef_df

,feature,coefficient
0,year,6306.441384
5382,type_truck,2286.840128
5361,fuel_gas,-2281.301684
3,odometer,-1899.975280
2,cylinders,1494.336750
...,...,...
4115,model_routan sel premium,-0.000000
4099,model_rogue sv suv,-0.000000
4103,model_romeo stelvio,-0.000000
4105,model_rondo ex,0.000000


In [24]:

# Features Lasso kept (non-zero)
selected = coef_df[coef_df['coefficient'] != 0]
print(f"Best alpha: {lasso.alpha_:.4f}")
print(f"Features kept: {len(selected)} / {len(X.columns)}")
print(selected.head(20))

Best alpha: 14.0255
Features kept: 3756 / 5446
                         feature  coefficient
0                           year  6306.441384
5382                  type_truck  2286.840128
5361                    fuel_gas -2281.301684
3                       odometer -1899.975280
2                      cylinders  1494.336750
404         manufacturer_ferrari  1435.888425
5371                   drive_fwd -1420.069006
4694  model_super duty f-550 drw  1320.047196
1                      condition  1288.085638
4284                 model_sedan  1244.347603
5380                 type_pickup  1185.602051
5137                 model_wagon  1095.605403
1704              model_corvette  1055.234698
3262                   model_k10  1012.645664
1719                 model_coupe   962.064743
425         manufacturer_porsche   818.697597
5362                 fuel_hybrid  -784.682110
4083              model_roadster   766.772274
5372                   drive_rwd  -743.352251
645                   model_3100 

In [25]:
# Get a clean view of what matters most
print("=== TOP POSITIVE (price drivers) ===")
print(selected[selected['coefficient'] > 0].head(10).to_string(index=False))

print("\n=== TOP NEGATIVE (price reducers) ===")
print(selected[selected['coefficient'] < 0].head(10).to_string(index=False))

=== TOP POSITIVE (price drivers) ===
                   feature  coefficient
                      year  6306.441384
                type_truck  2286.840128
                 cylinders  1494.336750
      manufacturer_ferrari  1435.888425
model_super duty f-550 drw  1320.047196
                 condition  1288.085638
               model_sedan  1244.347603
               type_pickup  1185.602051
               model_wagon  1095.605403
            model_corvette  1055.234698

=== TOP NEGATIVE (price reducers) ===
             feature  coefficient
            fuel_gas -2281.301684
            odometer -1899.975280
           drive_fwd -1420.069006
         fuel_hybrid  -784.682110
           drive_rwd  -743.352251
          type_sedan  -504.574849
 manufacturer_nissan  -466.807729
    manufacturer_kia  -332.771287
 manufacturer_subaru  -328.835906
title_status_rebuilt  -288.841186


In [26]:
sample = auto_X_test.iloc[[0]]  # double brackets keep it as DataFrame
predicted = auto_pipe.predict(sample)
actual = auto_y_test.iloc[0]
print(f"Predicted: ${predicted[0]:,.2f}")
print(f"Actual:    ${actual:,.2f}")
print(f"Diff:      ${abs(predicted[0] - actual):,.2f}")

NameError: name 'auto_X_test' is not defined